# Multi-UAV Cooperative 3D Navigation with Genuine MAPPO
### Standalone, Minimal, High-Quality Google Colab Implementation

This notebook implements a self-contained, lightweight Multi-Agent Reinforcement Learning (MARL) framework for **four cooperative UAVs** performing 3D waypoint navigation around obstacles in a bounded airspace.

**Core Features & Mathematical Guarantees:**
- **Centralized Training with Decentralized Execution (CTDE)**: Shared Actor policy $\pi_\theta(a_i|o_i)$ across all 4 UAVs and one Centralized Critic $V_\phi(s)$ observing joint fleet state.
- **Bounded Continuous Policy**: Tanh-squashed continuous Gaussian distribution with exact analytical **Jacobian transformation correction** on log-probabilities.
- **Pure Potential-Based Reward Shaping (PBRS)**: $F(s, s') = \gamma \Phi(s') - \Phi(s)$ with bounded non-negative potential $\Phi(s) = \frac{D_{\max} - d(s)}{10} \ge 0$ over the full 3D airspace diagonal ($300\,\text{m}$), guaranteeing zero cyclic reward harvesting.
- **GAE Consistency**: Distinguishes true terminal arrivals ($V(s_T)=0$) from artificial time-limit horizon truncations ($V(s_T)$ bootstrapped).
- **Zero External Dependencies**: Uses only standard Python libraries, NumPy, PyTorch, and Matplotlib. Runs out of the box in Google Colab (CPU or T4 GPU).


In [ ]:
# =============================================================================
# SECTION 1: IMPORTS, SEEDS, DEVICE, AND HYPERPARAMETERS
# =============================================================================
import os, math, random, csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Reproducibility & Device Selection
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Environment & Algorithm Hyperparameters
NUM_UAVS = 4
MAX_STEPS = 50           # Horizon per episode
TOTAL_EPISODES = 300     # Default training budget (set to e.g. 10 for quick smoke test)
AIRSPACE = np.array([200.0, 200.0, 100.0], dtype=np.float32)  # [X, Y, Z] bounds
D_MAX = float(np.linalg.norm(AIRSPACE))  # 300.0m diagonal (ensures Phi(s) >= 0 everywhere)
V_MAX = 1.5              # Max step displacement (m)
D_REACH = 5.0            # Goal arrival tolerance (m)
D_SAFE = 4.0             # Inter-UAV collision threshold (m)
OBSTACLES = [(100.0, 75.0, 12.0, 80.0), (100.0, 125.0, 12.0, 80.0)]  # (cx, cy, r, h)

# PPO Hyperparameters
GAMMA = 0.99             # Discount factor (matches PBRS shaping)
GAE_LAMBDA = 0.95        # GAE parameter
PPO_EPOCHS = 4           # Policy update epochs per rollout
PPO_CLIP = 0.2           # PPO clip ratio epsilon
LR_ACTOR = 3e-4
LR_CRITIC = 1e-3
ENTROPY_COEF = 0.01

print(f"[System] Device: {DEVICE} | Airspace: {AIRSPACE.tolist()}m (Diagonal: {D_MAX:.1f}m)")


In [ ]:
# =============================================================================
# SECTION 2: MINIMAL VECTORIZED 3D COOPERATIVE NAVIGATION ENVIRONMENT
# =============================================================================
class CooperativeNavigationEnv:
    """Vectorized 3D multi-UAV navigation task with cylindrical obstacles and PBRS rewards."""
    def __init__(self, num_uavs=NUM_UAVS, max_steps=MAX_STEPS):
        self.num_uavs, self.max_steps = num_uavs, max_steps
        self.obs_dim, self.action_dim = 18, 3
        self.state_dim = num_uavs * self.obs_dim  # 72 centralized critic state

    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.step_count = 0
        self.arrived = np.zeros(self.num_uavs, dtype=bool)
        center, r = np.array([100.0, 100.0, 50.0], dtype=np.float32), 45.0
        angles = np.array([0.0, np.pi/2, np.pi, 3*np.pi/2])
        self.positions = np.zeros((self.num_uavs, 3), dtype=np.float32)
        self.goals = np.zeros((self.num_uavs, 3), dtype=np.float32)

        # Perimeter crossing: UAVs start on circle, targets on opposite side
        for i in range(self.num_uavs):
            jitter = np.random.uniform(-4.0, 4.0, size=3)
            self.positions[i] = np.clip(center + [r*np.cos(angles[i]), r*np.sin(angles[i]), 0.0] + jitter, [5, 5, 10], AIRSPACE - 5)
            self.goals[i] = np.clip(center + [r*np.cos(angles[i] + np.pi), r*np.sin(angles[i] + np.pi), 0.0], [5, 5, 10], AIRSPACE - 5)

        self.prev_dists = np.linalg.norm(self.positions - self.goals, axis=-1)
        return self._get_obs(), self._get_state()

    def _get_obs(self):
        obs = np.zeros((self.num_uavs, self.obs_dim), dtype=np.float32)
        for i in range(self.num_uavs):
            p, g = self.positions[i], self.goals[i]
            peer_rels = [(self.positions[j] - p) / D_MAX for j in range(self.num_uavs) if j != i]
            nearest_obs = min(OBSTACLES, key=lambda o: np.linalg.norm(p[:2] - np.array(o[:2])))
            rel_obs = (np.array(nearest_obs[:2]) - p[:2]) / D_MAX
            obs[i] = np.concatenate([p / AIRSPACE, (g - p) / D_MAX, [np.linalg.norm(g - p) / D_MAX], np.concatenate(peer_rels), rel_obs])
        return obs

    def _get_state(self):
        return self._get_obs().flatten()

    def step(self, actions):
        self.step_count += 1
        actions = np.clip(actions, -V_MAX, V_MAX)
        for i in range(self.num_uavs):
            if not self.arrived[i]:
                self.positions[i] = np.clip(self.positions[i] + actions[i], [0, 0, 10], AIRSPACE)

        curr_dists = np.linalg.norm(self.positions - self.goals, axis=-1)
        rewards = np.zeros(self.num_uavs, dtype=np.float32)
        inter_cols, obs_cols = 0, 0

        # Potential-Based Reward Shaping: F = gamma * Phi(s') - Phi(s), Phi(s) = (D_MAX - d) / 10 >= 0
        shaping = 4.0 * (GAMMA * ((D_MAX - curr_dists) / 10.0) - ((D_MAX - self.prev_dists) / 10.0))
        for i in range(self.num_uavs):
            if not self.arrived[i]:
                rewards[i] += shaping[i] - 0.02 * float(np.sum(actions[i]**2))
                if curr_dists[i] <= D_REACH:
                    self.arrived[i] = True
                    rewards[i] += 50.0  # Terminal arrival milestone

        # Collision detection (peer-to-peer and obstacle cylinders)
        for i in range(self.num_uavs):
            for j in range(i + 1, self.num_uavs):
                if np.linalg.norm(self.positions[i] - self.positions[j]) < D_SAFE:
                    rewards[i] -= 10.0; rewards[j] -= 10.0; inter_cols += 1
            for cx, cy, r_c, h_c in OBSTACLES:
                if np.linalg.norm(self.positions[i][:2] - [cx, cy]) < r_c and self.positions[i][2] < h_c:
                    rewards[i] -= 10.0; obs_cols += 1

        all_arrived = bool(np.all(self.arrived))
        is_truncated = bool(self.step_count >= self.max_steps and not all_arrived)
        if is_truncated:
            rewards -= 20.0 * (~self.arrived)  # Timeout penalty

        self.prev_dists = curr_dists.copy()
        info = {"arrivals": int(np.sum(self.arrived)), "collisions": inter_cols + obs_cols, "mean_goal_dist": float(np.mean(curr_dists))}
        return self._get_obs(), self._get_state(), rewards, all_arrived, is_truncated, info


In [ ]:
# =============================================================================
# SECTION 3: GENUINE MAPPO - SHARED ACTOR, CENTRALIZED CRITIC & GAE
# =============================================================================
class SharedActor(nn.Module):
    """Shared policy with Tanh-squashed Gaussian and analytical Jacobian correction."""
    def __init__(self, obs_dim=18, action_dim=3, hidden_dim=64, v_max=V_MAX):
        super().__init__()
        self.v_max = v_max
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, action_dim)
        )
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, obs):
        return self.net(obs), torch.exp(torch.clamp(self.log_std, -2.0, 0.5))

    def sample(self, obs):
        mu, std = self.forward(obs)
        dist = Normal(mu, std)
        u = dist.rsample()  # Latent Gaussian action
        action = self.v_max * torch.tanh(u)
        # Exact Jacobian: log pi(a) = log N(u) - sum(log(1 - tanh(u)^2 + 1e-6)) - 3*log(v_max)
        jacobian = torch.log(1.0 - torch.tanh(u).pow(2) + 1e-6).sum(dim=-1) + 3.0 * math.log(self.v_max)
        log_prob = dist.log_prob(u).sum(dim=-1) - jacobian
        return action, u, log_prob

    def evaluate(self, obs, latent_u):
        mu, std = self.forward(obs)
        dist = Normal(mu, std)
        jacobian = torch.log(1.0 - torch.tanh(latent_u).pow(2) + 1e-6).sum(dim=-1) + 3.0 * math.log(self.v_max)
        log_prob = dist.log_prob(latent_u).sum(dim=-1) - jacobian
        return log_prob, dist.entropy().sum(dim=-1)

class CentralizedCritic(nn.Module):
    """Centralized value network taking joint state and returning per-UAV values."""
    def __init__(self, state_dim=72, num_uavs=NUM_UAVS, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, num_uavs)
        )

    def forward(self, state):
        return self.net(state)

def compute_gae(rewards, values, terminated, truncated, gamma=GAMMA, gae_lambda=GAE_LAMBDA):
    """GAE distinguishing terminal arrival (bootstrap=0) from time horizon truncation (bootstrap=V(s_T))."""
    T = len(rewards)
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_gae = np.zeros(NUM_UAVS, dtype=np.float32)
    for t in reversed(range(T)):
        next_non_terminal = 0.0 if (t == T - 1 and terminated) else 1.0
        delta = rewards[t] + gamma * values[t + 1] * next_non_terminal - values[t]
        last_gae = delta + gamma * gae_lambda * next_non_terminal * last_gae
        advantages[t] = last_gae
    return advantages, advantages + values[:-1]


In [ ]:
# =============================================================================
# SECTION 4: TRAINING MAPPO ON 4-UAV COOPERATIVE NAVIGATION
# =============================================================================
env = CooperativeNavigationEnv()
actor = SharedActor().to(DEVICE)
critic = CentralizedCritic().to(DEVICE)
opt_actor = optim.Adam(actor.parameters(), lr=LR_ACTOR)
opt_critic = optim.Adam(critic.parameters(), lr=LR_CRITIC)

train_history = {"episode": [], "return": [], "arrival_rate": [], "collisions": [], "mean_goal_dist": []}
best_arr = -1.0

print(f"[Training] Starting MAPPO training for {TOTAL_EPISODES} episodes on {DEVICE}...")
for ep in range(1, TOTAL_EPISODES + 1):
    obs, state = env.reset(seed=ep + 100)
    ep_obs, ep_state, ep_latent, ep_logp, ep_rew, ep_val = [], [], [], [], [], []
    ep_cols = 0

    for step in range(MAX_STEPS):
        with torch.no_grad():
            act_t, u_t, logp_t = actor.sample(torch.tensor(obs, dtype=torch.float32, device=DEVICE))
            val_t = critic(torch.tensor(state, dtype=torch.float32, device=DEVICE))

        next_obs, next_state, rew, term, trunc, info = env.step(act_t.cpu().numpy())
        ep_obs.append(obs); ep_state.append(state); ep_latent.append(u_t.cpu().numpy())
        ep_logp.append(logp_t.cpu().numpy()); ep_rew.append(rew); ep_val.append(val_t.cpu().numpy())
        ep_cols += info["collisions"]
        obs, state = next_obs, next_state
        if term or trunc: break

    with torch.no_grad():
        ep_val.append(critic(torch.tensor(state, dtype=torch.float32, device=DEVICE)).cpu().numpy())

    adv, rets = compute_gae(np.array(ep_rew), np.array(ep_val), term, trunc)
    flat_obs = torch.tensor(np.array(ep_obs).reshape(-1, env.obs_dim), dtype=torch.float32, device=DEVICE)
    flat_u = torch.tensor(np.array(ep_latent).reshape(-1, env.action_dim), dtype=torch.float32, device=DEVICE)
    flat_logp = torch.tensor(np.array(ep_logp).reshape(-1), dtype=torch.float32, device=DEVICE)
    flat_adv = torch.tensor(adv.reshape(-1), dtype=torch.float32, device=DEVICE)
    flat_adv = (flat_adv - flat_adv.mean()) / (flat_adv.std() + 1e-8)  # Normalized advantages
    flat_state = torch.tensor(np.array(ep_state), dtype=torch.float32, device=DEVICE)
    flat_rets = torch.tensor(rets, dtype=torch.float32, device=DEVICE)

    # MAPPO PPO Minibatch Updates
    for _ in range(PPO_EPOCHS):
        new_logp, entropy = actor.evaluate(flat_obs, flat_u)
        ratio = torch.exp(new_logp - flat_logp)
        actor_loss = -torch.min(ratio * flat_adv, torch.clamp(ratio, 1 - PPO_CLIP, 1 + PPO_CLIP) * flat_adv).mean() - ENTROPY_COEF * entropy.mean()
        opt_actor.zero_grad(); actor_loss.backward(); nn.utils.clip_grad_norm_(actor.parameters(), 0.5); opt_actor.step()

        critic_loss = nn.MSELoss()(critic(flat_state), flat_rets)
        opt_critic.zero_grad(); critic_loss.backward(); nn.utils.clip_grad_norm_(critic.parameters(), 0.5); opt_critic.step()

    arr = (info["arrivals"] / NUM_UAVS) * 100.0
    for k, v in [("episode", ep), ("return", float(np.sum(ep_rew))), ("arrival_rate", arr), ("collisions", ep_cols), ("mean_goal_dist", info["mean_goal_dist"])]:
        train_history[k].append(v)

    if arr > best_arr:
        best_arr = arr
        torch.save({"actor": actor.state_dict(), "critic": critic.state_dict()}, "best_mappo_colab_agent.pt")

    if ep % 20 == 0 or ep == TOTAL_EPISODES:
        print(f"Ep {ep:3d}/{TOTAL_EPISODES} | Ret: {np.mean(train_history['return'][-20:]):7.1f} | Arr: {np.mean(train_history['arrival_rate'][-20:]):5.1f}% | Col: {np.mean(train_history['collisions'][-20:]):4.1f}")

print(f"[Training Complete] Peak Checkpoint Arrival Rate: {best_arr:.1f}%")


In [ ]:
# =============================================================================
# SECTION 5: LEARNING CURVES
# =============================================================================
def moving_avg(data, window=15):
    w = max(1, min(window, len(data)))
    return np.convolve(data, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = [("return", "Episode Return", "tab:blue"), ("arrival_rate", "Goal Arrival Rate (%)", "tab:green"), ("collisions", "Collision Count", "tab:red")]
for ax, (key, title, color) in zip(axes, titles):
    raw = train_history[key]
    ma = moving_avg(raw)
    ax.plot(train_history["episode"], raw, alpha=0.3, color=color, label="Raw")
    ax.plot(train_history["episode"][-len(ma):], ma, color=color, lw=2, label="MA")
    ax.set_title(title); ax.set_xlabel("Episode"); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()


In [ ]:
# =============================================================================
# SECTION 6: EVALUATION AGAINST HEURISTIC BASELINE & 3D TRAJECTORIES
# =============================================================================
def heuristic_controller(positions, goals):
    """Direct goal attraction with inter-UAV and obstacle avoidance."""
    actions = np.zeros((NUM_UAVS, 3), dtype=np.float32)
    for i in range(NUM_UAVS):
        vec = goals[i] - positions[i]
        d = np.linalg.norm(vec)
        v = (vec / d * V_MAX) if d > 1e-4 else np.zeros(3)
        # Peer repulsion
        for j in range(NUM_UAVS):
            if i != j and 0 < np.linalg.norm(positions[i] - positions[j]) < (D_SAFE + 3.0):
                v += (positions[i] - positions[j]) / (np.linalg.norm(positions[i] - positions[j]) + 1e-6) * (V_MAX * 0.8)
        # Obstacle avoidance
        for cx, cy, r_c, h_c in OBSTACLES:
            if np.linalg.norm(positions[i][:2] - [cx, cy]) < r_c + 5.0 and positions[i][2] < h_c:
                diff = positions[i][:2] - [cx, cy]
                v[:2] += diff / (np.linalg.norm(diff) + 1e-6) * V_MAX
        actions[i] = np.clip(v, -V_MAX, V_MAX)
    return actions

EVAL_SEEDS = list(range(1000, 1030))
eval_results = []
for mode, name in [(actor, "MAPPO"), (None, "Heuristic")]:
    arr_l, col_l, dist_l, ret_l = [], [], [], []
    for s in EVAL_SEEDS:
        obs, state = env.reset(seed=s)
        ep_ret, ep_col = 0.0, 0
        for _ in range(MAX_STEPS):
            if mode is not None:
                with torch.no_grad():
                    a, _, _ = actor.sample(torch.tensor(obs, dtype=torch.float32, device=DEVICE))
                    act = a.cpu().numpy()
            else:
                act = heuristic_controller(env.positions, env.goals)
            obs, state, rew, term, trunc, info = env.step(act)
            ep_ret += float(np.sum(rew)); ep_col += info["collisions"]
            if term or trunc: break
        arr = (info["arrivals"] / NUM_UAVS) * 100.0
        arr_l.append(arr); col_l.append(ep_col); dist_l.append(info["mean_goal_dist"]); ret_l.append(ep_ret)
        eval_results.append({"controller": name, "seed": s, "arrival_rate": arr, "collisions": ep_col, "goal_dist": info["mean_goal_dist"], "return": ep_ret})
    print(f"\n=== {name} (30 Held-Out Scenarios) ===\nArrival: {np.mean(arr_l):5.1f}% ± {np.std(arr_l):4.1f}% | Collisions: {np.mean(col_l):5.2f} ± {np.std(col_l):4.2f} | Goal Dist: {np.mean(dist_l):5.1f}m | Return: {np.mean(ret_l):6.1f}")

# 3D Sample Trajectory Plot
obs, state = env.reset(seed=1001)
traj = [env.positions.copy()]
for _ in range(MAX_STEPS):
    with torch.no_grad():
        a, _, _ = actor.sample(torch.tensor(obs, dtype=torch.float32, device=DEVICE))
    obs, _, _, term, trunc, _ = env.step(a.cpu().numpy())
    traj.append(env.positions.copy())
    if term or trunc: break
traj = np.array(traj)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:purple']
for i in range(NUM_UAVS):
    ax.plot(traj[:, i, 0], traj[:, i, 1], traj[:, i, 2], color=colors[i], lw=2, label=f"UAV {i}")
    ax.scatter(traj[0, i, 0], traj[0, i, 1], traj[0, i, 2], color=colors[i], marker="o", s=50)
    ax.scatter(env.goals[i, 0], env.goals[i, 1], env.goals[i, 2], color=colors[i], marker="*", s=100)
for cx, cy, r_c, h_c in OBSTACLES:
    z_g = np.linspace(0, h_c, 8); th_g = np.linspace(0, 2*np.pi, 16); th_m, z_m = np.meshgrid(th_g, z_g)
    ax.plot_surface(cx + r_c*np.cos(th_m), cy + r_c*np.sin(th_m), z_m, color='gray', alpha=0.4)
ax.set_title("Sample 4-UAV Cooperative 3D Trajectories (MAPPO)")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
ax.legend()
plt.tight_layout()
plt.savefig("sample_3d_trajectories.png", dpi=150)
plt.show()


In [ ]:
# =============================================================================
# SECTION 7: EXPORT CHECKPOINT, RESULTS CSV & GOOGLE COLAB DOWNLOAD HELPER
# =============================================================================
csv_path = "evaluation_results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["controller", "seed", "arrival_rate", "collisions", "goal_dist", "return"])
    writer.writeheader(); writer.writerows(eval_results)

model_path = "mappo_uav_final_model.pt"
torch.save({"actor": actor.state_dict(), "critic": critic.state_dict()}, model_path)
print(f"[Export] Saved artifacts: {csv_path}, {model_path}")

try:
    from google.colab import files
    for f in [model_path, csv_path, "training_curves.png", "sample_3d_trajectories.png"]:
        if os.path.exists(f): files.download(f)
except ImportError:
    pass


In [ ]:
# =============================================================================
# INLINE SMOKE VERIFICATION: DIMENSIONS, BOUNDS, FINITE OUTPUTS
# =============================================================================
test_env = CooperativeNavigationEnv()
t_obs, t_state = test_env.reset(seed=999)
assert t_obs.shape == (NUM_UAVS, 18) and t_state.shape == (72,)
with torch.no_grad():
    t_act, t_u, t_logp = actor.sample(torch.tensor(t_obs, dtype=torch.float32, device=DEVICE))
assert t_act.shape == (NUM_UAVS, 3) and bool(torch.all(t_act.abs() <= V_MAX + 1e-5)) and bool(torch.isfinite(t_logp).all())
_, _, t_rew, _, _, _ = test_env.step(t_act.cpu().numpy())
assert t_rew.shape == (NUM_UAVS,)
print("[Smoke Verification PASS] Environment, policy bounds, and outputs verified!")
